In [1]:
%load_ext sql
%sql sqlite://

In [ ]:
%%time
import csv
import sqlite3

%sql sqlite:///clasher.db
    
conn = sqlite3.connect("clasher.db")
cursor = conn.cursor()

with open('battlesStaging_12072020_to_12262020_WL_tagged.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute(f"DROP TABLE IF EXISTS clasher")
    cursor.execute(f"CREATE TABLE clasher ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO clasher VALUES ({placeholders})', row)

    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

In [ ]:
cursor.execute("SELECT * FROM clasher")
print([desc[0] for desc in cursor.description])

In [ ]:
with open('CardMasterListSeason18_12082020.csv', 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames
    
    # Create table
    col_defs = ", ".join([f'"{col}" TEXT' for col in columns])
    cursor.execute("DROP TABLE IF EXISTS card_names")
    cursor.execute(f"CREATE TABLE card_names ({col_defs})")
    
    # Insert all rows
    placeholders = ", ".join(["?" for _ in columns])
    rows = [tuple(row[col] for col in columns) for row in reader]
    
    for row in rows:
        cursor.execute(f'INSERT INTO card_names VALUES ({placeholders})', row)
    conn.commit()

print("Done! Rows loaded:", cursor.execute("SELECT COUNT(*) FROM card_names").fetchone()[0])
print("First row:", cursor.execute("SELECT * FROM card_names").fetchone())
print("Columns:", cursor.execute("SELECT * FROM card_names").description)

In [ ]:
%%time
cursor.execute("""
    CREATE TABLE clasher_new AS
    SELECT c.*,
        w1."team.card1.name" as "winner.card1.name",
        w2."team.card1.name" as "winner.card2.name",
        w3."team.card1.name" as "winner.card3.name",
        w4."team.card1.name" as "winner.card4.name",
        w5."team.card1.name" as "winner.card5.name",
        w6."team.card1.name" as "winner.card6.name",
        w7."team.card1.name" as "winner.card7.name",
        w8."team.card1.name" as "winner.card8.name",
        l1."team.card1.name" as "loser.card1.name",
        l2."team.card1.name" as "loser.card2.name",
        l3."team.card1.name" as "loser.card3.name",
        l4."team.card1.name" as "loser.card4.name",
        l5."team.card1.name" as "loser.card5.name",
        l6."team.card1.name" as "loser.card6.name",
        l7."team.card1.name" as "loser.card7.name",
        l8."team.card1.name" as "loser.card8.name"
    FROM clasher c
    LEFT JOIN card_names w1 ON c."winner.card1.id" = w1."team.card1.id"
    LEFT JOIN card_names w2 ON c."winner.card2.id" = w2."team.card1.id"
    LEFT JOIN card_names w3 ON c."winner.card3.id" = w3."team.card1.id"
    LEFT JOIN card_names w4 ON c."winner.card4.id" = w4."team.card1.id"
    LEFT JOIN card_names w5 ON c."winner.card5.id" = w5."team.card1.id"
    LEFT JOIN card_names w6 ON c."winner.card6.id" = w6."team.card1.id"
    LEFT JOIN card_names w7 ON c."winner.card7.id" = w7."team.card1.id"
    LEFT JOIN card_names w8 ON c."winner.card8.id" = w8."team.card1.id"
    LEFT JOIN card_names l1 ON c."loser.card1.id" = l1."team.card1.id"
    LEFT JOIN card_names l2 ON c."loser.card2.id" = l2."team.card1.id"
    LEFT JOIN card_names l3 ON c."loser.card3.id" = l3."team.card1.id"
    LEFT JOIN card_names l4 ON c."loser.card4.id" = l4."team.card1.id"
    LEFT JOIN card_names l5 ON c."loser.card5.id" = l5."team.card1.id"
    LEFT JOIN card_names l6 ON c."loser.card6.id" = l6."team.card1.id"
    LEFT JOIN card_names l7 ON c."loser.card7.id" = l7."team.card1.id"
    LEFT JOIN card_names l8 ON c."loser.card8.id" = l8."team.card1.id"
""")

# 4. Replace clasher with the new table
cursor.execute("DROP TABLE clasher")
cursor.execute("ALTER TABLE clasher_new RENAME TO clasher")
conn.commit()

print("Done!", cursor.execute("SELECT COUNT(*) FROM clasher").fetchone()[0], "rows")
print("First row:", cursor.execute("SELECT * FROM clasher").fetchone())

In [ ]:
%%time
cursor.execute("DROP TABLE IF EXISTS card_stats")
cursor.execute("""
    CREATE TABLE card_stats AS
    SELECT card_name, SUM(wins) as win_count, SUM(losses) as loss_count
    FROM (
        SELECT "winner.card1.name" as card_name, 1 as wins, 0 as losses FROM clasher UNION ALL
        SELECT "winner.card2.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card3.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card4.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card5.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card6.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card7.name", 1, 0 FROM clasher UNION ALL
        SELECT "winner.card8.name", 1, 0 FROM clasher UNION ALL
        SELECT "loser.card1.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card2.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card3.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card4.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card5.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card6.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card7.name", 0, 1 FROM clasher UNION ALL
        SELECT "loser.card8.name", 0, 1 FROM clasher
    )
    WHERE card_name IS NOT NULL
    GROUP BY card_name
    ORDER BY win_count DESC
""")
conn.commit()

cursor.execute("SELECT * FROM card_stats LIMIT 10").fetchall()

In [ ]:
%%time
cursor.execute("ALTER TABLE card_stats ADD COLUMN win_ratio REAL")
cursor.execute("UPDATE card_stats SET win_ratio = ROUND(win_count * 1.0 / (win_count + loss_count), 3)")
conn.commit()

In [ ]:
results = cursor.execute("""
    SELECT * FROM card_stats 
    ORDER BY win_ratio DESC 
""").fetchall()

for row in results:
    print(row)